# Parameter Sensitivity Explorer (Vectorized)

This notebook performs a **vectorized grid search** using `vectorbt` to analyze parameter sensitivity for the trend-following strategy.

**Key Difference from Optuna:**
- **Vectorization**: Simulates ALL parameter combinations simultaneously (thousands of backtests in seconds).
- **Heatmaps**: Visualizes the "landscape" of profitability to find robust parameter regions, not just a single peak.
- **Core**: Uses `FastBacktest` engine.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import plotly.graph_objects as go

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.utils.setup import load_data_and_setup
from ggTrader.core.fast_backtest import FastBacktest

In [ ]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": None,  # Set to a list like ["BTC", "ETH"] to override JSON
    "SYMBOLS_FILE": "data/top_50_consistent_movers.json",
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-06-01",
    "INTERVAL": "4h",
    "START_CASH": 10000,
    "PORTFOLIO_SHARE": 0.20,
}

print("Configuration loaded.")

In [ ]:
print("Loading data...")
try:
    ohlcv = load_data_and_setup(CONSTANTS)
    print(f"Loaded {len(ohlcv)} rows for {len(ohlcv.columns.levels[0])} symbols.")
except Exception as e:
    print(f"Error loading data: {e}")

In [ ]:
# --- Define Parameter Grid ---
# vectorbt will create a Cartesian product of these lists
params = {
    "adx_threshold": list(range(15, 45, 5)),      # [15, 20, ..., 40]
    "adx_length": [14],                           # Fixed
    "sar_acceleration": [0.02, 0.03],             # 2 options
    "sar_maximum": [0.2],                         # Fixed
    "atr_multiplier": list(np.arange(2.0, 6.0, 0.5)),  # [2.0, 2.5, ..., 5.5]
    "atr_length": [14],                           # Fixed
    "use_dmp_cross": [True, False],               # Boolean toggle
}

# Calculate total combinations
total_combos = 1
for k, v in params.items():
    if isinstance(v, list):
        total_combos *= len(v)
print(f"Testing {total_combos} parameter combinations across {len(ohlcv.columns.levels[0])} symbols...")

In [ ]:
# --- Run Vectorized Backtest ---
print("Running FastBacktest...")
# FastBacktest handles SignalFactory.run(param_product=True) internally
engine = FastBacktest(ohlcv, params)
pf = engine.run()

print("Backtest Complete.")

In [ ]:
# --- Analyze Metrics ---
# Calculate Sharpe Ratio for every combination (aggregated across symbols?)
# pf contains MultiIndex columns: (param1, param2, ..., symbol)

# Group by Parameters to see which params work best *on average* across all symbols
param_names = list(params.keys())
metrics = pf.sharpe_ratio(group_by=param_names)

# Convert to DataFrame for easier plotting
results_df = metrics.reset_index()
results_df.rename(columns={0: "Sharpe Ratio"}, inplace=True)

print("Top 5 Parameter Sets:")
print(results_df.sort_values("Sharpe Ratio", ascending=False).head(5))

In [ ]:
# --- Heatmap Visualization ---
# Let's visualize the interaction between ADX Threshold and ATR Multiplier
# We fix other parameters to their best values (or arbitrary ones) to slice the data

# Pivot table for heatmap
heatmap_data = results_df.pivot_table(
    index="atr_multiplier", 
    columns="adx_threshold", 
    values="Sharpe Ratio",
    aggfunc="mean" # In case multiple other params map to this cell, take mean
)

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='Viridis',
    colorbar=dict(title='Sharpe Ratio')
))

fig.update_layout(
    title="Sharpe Ratio Heatmap: ATR Multiplier vs ADX Threshold",
    xaxis_title="ADX Threshold",
    yaxis_title="ATR Multiplier"
)

fig.show()

In [ ]:
# --- Robustness Check ---
# Parameter stability: Does a parameter perform well over a RANGE of values?
# Box plot for Boolean parameter 'use_dmp_cross'

import plotly.express as px

fig2 = px.box(results_df, x="use_dmp_cross", y="Sharpe Ratio", title="Impact of DMP Cross Filter")
fig2.show()